                                       End-to-End PySpark Project
                                   Movie Ratings Analytics Platform



Project Title - Scalable Movie Ratings Analytics Pipeline Using PySpark





Project Overview

Business Problem

A movie streaming company collects millions of user ratings daily.
The analytics team wants to:

- Analyse user behaviour
- Identify trending/popular movies
- Calculate average ratings
- Detect low-quality or corrupt records
- Generate reporting datasets for dashboards
- Handle incremental incoming data efficiently

As a Data Engineer, your responsibility is to build a production-grade PySpark ETL pipeline that:

- Reads raw ratings data
- Cleans and validates records
- Handles null and duplicate values
- Joins business reference tables
- Performs aggregations
- Generates analytical outputs
- Stores processed data efficiently


 Dataset Details

 Ratings Dataset
Column	Description
user_id	Unique user ID
movie_id	Unique movie ID
rating	Rating from 1-5
timestamp	Rating timestamp


Movie Details Dataset

Column	Description
movie_id	Unique movie ID
movie_name	Movie name
genre	Genre
release_year	Release year



In [0]:
ratings_data = [
    (1, 101, 5, "2026-01-01 10:00:00"),
    (2, 101, 4, "2026-01-01 10:05:00"),
    (3, 102, 3, "2026-01-01 11:00:00"),
    (4, 103, None, "2026-01-01 11:30:00"),   # corrupt (null rating)
    (5, 104, 6, "2026-01-01 12:00:00"),      # invalid rating (>5)
    (1, 101, 5, "2026-01-01 10:00:00"),      # duplicate
    (6, 105, 2, None),                       # missing timestamp
]

ratings_columns = ["user_id", "movie_id", "rating", "timestamp"]

ratings_df = spark.createDataFrame(ratings_data, ratings_columns)
ratings_df.show()

+-------+--------+------+-------------------+
|user_id|movie_id|rating|          timestamp|
+-------+--------+------+-------------------+
|      1|     101|     5|2026-01-01 10:00:00|
|      2|     101|     4|2026-01-01 10:05:00|
|      3|     102|     3|2026-01-01 11:00:00|
|      4|     103|  NULL|2026-01-01 11:30:00|
|      5|     104|     6|2026-01-01 12:00:00|
|      1|     101|     5|2026-01-01 10:00:00|
|      6|     105|     2|               NULL|
+-------+--------+------+-------------------+



In [0]:
movies_data = [
    (101, "Avengers", "Action", 2012),
    (102, "Titanic", "Romance", 1997),
    (103, "Inception", "Sci-Fi", 2010),
    (104, "Joker", "Drama", 2019),
    (105, "Interstellar", "Sci-Fi", 2014),
]

movies_columns = ["movie_id", "movie_name", "genre", "release_year"]

movies_df = spark.createDataFrame(movies_data, movies_columns)
movies_df.show()

+--------+------------+-------+------------+
|movie_id|  movie_name|  genre|release_year|
+--------+------------+-------+------------+
|     101|    Avengers| Action|        2012|
|     102|     Titanic|Romance|        1997|
|     103|   Inception| Sci-Fi|        2010|
|     104|       Joker|  Drama|        2019|
|     105|Interstellar| Sci-Fi|        2014|
+--------+------------+-------+------------+



In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col
from pyspark.sql.functions import *

In [0]:
#DATA CLEANING & VALIDATION
# Remove NULLs
ratings_clean = ratings_df.dropna(subset=["rating", "timestamp"])

In [0]:
#Remove Invalid Ratings (1–5 only)

ratings_clean = ratings_clean.filter((col("rating") >= 1) & (col("rating") <= 5))

In [0]:
#Remove Duplicates
ratings_clean = ratings_clean.dropDuplicates(["user_id", "movie_id", "timestamp"])

In [0]:
#Cleaned Ratings Output
ratings_clean.show()

+-------+--------+------+-------------------+
|user_id|movie_id|rating|          timestamp|
+-------+--------+------+-------------------+
|      1|     101|     5|2026-01-01 10:00:00|
|      2|     101|     4|2026-01-01 10:05:00|
|      3|     102|     3|2026-01-01 11:00:00|
+-------+--------+------+-------------------+



In [0]:
#JOIN MOVIE DETAILS
enriched_df = ratings_clean.join(movies_df, "movie_id", "left")
enriched_df.show()

+--------+-------+------+-------------------+----------+-------+------------+
|movie_id|user_id|rating|          timestamp|movie_name|  genre|release_year|
+--------+-------+------+-------------------+----------+-------+------------+
|     101|      1|     5|2026-01-01 10:00:00|  Avengers| Action|        2012|
|     101|      2|     4|2026-01-01 10:05:00|  Avengers| Action|        2012|
|     102|      3|     3|2026-01-01 11:00:00|   Titanic|Romance|        1997|
+--------+-------+------+-------------------+----------+-------+------------+



In [0]:
#ANALYTICS / AGGREGATION: Average Rating per Movie
avg_rating = enriched_df.groupBy("movie_id", "movie_name") \
    .agg(
        avg("rating").alias("avg_rating"),
        count("rating").alias("total_ratings")
    ) \
    .orderBy(desc("avg_rating"))

avg_rating.show()

+--------+----------+----------+-------------+
|movie_id|movie_name|avg_rating|total_ratings|
+--------+----------+----------+-------------+
|     101|  Avengers|       4.5|            2|
|     102|   Titanic|       3.0|            1|
+--------+----------+----------+-------------+



In [0]:
#Trending Movies (Most Rated)
trending = enriched_df.groupBy("movie_name") \
    .count() \
    .orderBy(desc("count"))

trending.show()

+----------+-----+
|movie_name|count|
+----------+-----+
|  Avengers|    2|
|   Titanic|    1|
+----------+-----+



In [0]:
#User Behavior (Most Active Users)
user_activity = enriched_df.groupBy("user_id") \
    .count() \
    .orderBy(desc("count"))

user_activity.show()

+-------+-----+
|user_id|count|
+-------+-----+
|      1|    1|
|      2|    1|
|      3|    1|
+-------+-----+



In [0]:
#Genre Insights
genre_analysis = enriched_df.groupBy("genre") \
    .agg(avg("rating").alias("avg_rating"))

genre_analysis.show()

+-------+----------+
|  genre|avg_rating|
+-------+----------+
| Action|       4.5|
|Romance|       3.0|
+-------+----------+



In [0]:
#FINAL REPORTING TABLE (FOR DASHBOARDS)
final_output = enriched_df.select(
    "user_id",
    "movie_name",
    "genre",
    "rating",
    "release_year",
    "timestamp"
)

final_output.show()

+-------+----------+-------+------+------------+-------------------+
|user_id|movie_name|  genre|rating|release_year|          timestamp|
+-------+----------+-------+------+------------+-------------------+
|      1|  Avengers| Action|     5|        2012|2026-01-01 10:00:00|
|      2|  Avengers| Action|     4|        2012|2026-01-01 10:05:00|
|      3|   Titanic|Romance|     3|        1997|2026-01-01 11:00:00|
+-------+----------+-------+------+------------+-------------------+



In [0]:
#SAVE PROCESSED DATA (OPTIMIZED FORMAT)
final_output.write.mode("overwrite") \
    .parquet("/Volumes/workspace/default/data2")